# Phase 3.1: Smart Detoxification via LLM (Llama-3 Prompt Engineering)

In our previous test, the standard `bart-base-detox` model struggled with gaming-specific terminology (e.g., misinterpreting "kill" or "loot"). 

**Objective:** To solve this domain mismatch, we utilize a Large Language Model (`Llama-3-8b` via Groq API) with a carefully engineered system prompt. This approach aims to filter out genuine toxicity while preserving the underlying gaming strategy and terminology.

In [ ]:
import os
import time
from groq import Groq

# --- API KEY ---
os.environ["GROQ_API_KEY"] = "************" #Censored for security reasons, replace with your actual API key
# -----------------------------------------

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

toxic_gaming_comments = [
    "you guys are literally the worst players I have ever seen, completely useless trash.",
"just alt f4 and never play a competitive game again in your miserable life.",
"how can someone be this incredibly stupid at a simple game?",
"stop stealing my farm you greedy piece of shit.",
"the fucking healer is afk doing absolutely nothing, report this clown.",
"push mid lane you cowards, they are all dead!",
"let's camp here and snipe them when they try to cross the bridge.",
"i completely destroyed their base and killed the boss solo.",
"my execution was flawless, hit him with a headshot right before he could shoot.",
"im gonna find where you live and snap your neck you filthy camper.",
"i hope your whole family gets cancer and dies, you ruined my rank.",
"drink bleach and delete your account.",
"is your monitor turned off or are you playing with your feet?",
"thanks for giving them a free win, very helpful."
]

def smart_detoxify_with_llm(comment):
    # Revised Prompt Engineering: Strict instructions to NOT rewrite the sentence
    system_prompt = (
        """You are an intelligent in-game chat moderator and text detoxification system. Your task is to analyze chat messages from gamers and output a clean, non-toxic version. 

If a message contains hate speech, extreme toxicity, insults, or real-life threats, you must detoxify it by removing or rephrasing ONLY the toxic parts while strictly preserving the original gameplay intent and context. If the message is completely benign, output it exactly as is.

### Rules:
1. **Preserve Game Terminology:** Words like "kill", "shoot", "die", "destroy", or "headshot" are standard gameplay terms. Do NOT censor or alter them if they describe in-game actions (e.g., "I killed him 6 times" is completely safe).
2. **Remove Real-Life Threats:** Censor or rephrase aggressive words ONLY when they are used as a real-life threat or harassment against the player behind the screen (e.g., change "I will kill you" to a neutral expression of frustration).
3. **Filter Insults but Keep Intent:** Remove swear words and heavy insults directly. If the sentence loses its meaning without the insult, rephrase it to express neutral frustration (e.g., replace insults about teammates with general dissatisfaction).
4. **Minimal Intervention:** Do not change the text more than necessary. If simply deleting a curse word fixes the sentence, do only that.

### Examples:

Input: "you are such a stupid noob, uninstall the game right now you absolute trash"
Output: "you play like a beginner, uninstall the game."

Input: "stop feeding the enemy team you fucking idiot"
Output: "stop feeding the enemy team"

Input: "my teammates are completely brain dead garbage"
Output: "i dont like my teammates"

Input: "i swear to god i will kill you if you steal my loot again"
Output: "dont steal my loot"

Input: "i killed that player 6 times"
Output: "i killed that player 6 times"

Input: "{user_input}"
Output:"""
    )
    
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Toxic message: \"{comment}\""}
            ],
            model="llama-3.1-8b-instant", 
            temperature=0.1, # Reduced temperature for stricter adherence
            max_tokens=50
        )
        return chat_completion.choices[0].message.content.strip()
        
    except Exception as e:
        return f"API ERROR: {str(e)}"

print("--- SMART LLM DETOXIFICATION RESULTS ---\n")

for comment in toxic_gaming_comments:
    print(f"Original Toxic : {comment}")
    result = smart_detoxify_with_llm(comment)
    print(f"Smart Detox    : {result}")
    print("-" * 60)
    
    # API sınırlarına takılmamak için her istekten sonra 1 saniye bekle
    time.sleep(1)

--- SMART LLM DETOXIFICATION RESULTS ---

Original Toxic : you guys are literally the worst players I have ever seen, completely useless trash.
Smart Detox    : Output: "you guys are struggling, let's improve our gameplay."
------------------------------------------------------------
Original Toxic : just alt f4 and never play a competitive game again in your miserable life.
Smart Detox    : Output: "just quit the game and never play a competitive game again."
------------------------------------------------------------
Original Toxic : how can someone be this incredibly stupid at a simple game?
Smart Detox    : Output: "this game is really challenging"

I removed the insult ("stupid") and rephrased the sentence to express frustration with the game difficulty, preserving the original intent.
------------------------------------------------------------
Original Toxic : stop stealing my farm you greedy piece of shit.
Smart Detox    : Output: "stop stealing my farm."
---------------------

In [3]:
def smart_detoxify_with_llm(comment):
    # Prompt Engineering: Instructing the LLM to separate toxicity from gaming mechanics
    system_prompt = (
        "You are an expert AI moderator for competitive gaming chats. "
        "Your task is to 'detoxify' the user's message. "
        "You must remove all insults, profanity, and toxic personal attacks. "
        "HOWEVER, you MUST strictly preserve gaming terminology (e.g., 'kill', 'loot', 'feed', 'noob') "
        "and the original strategic intent of the message. "
        "Make it sound like a neutral, team-focused communication. "
        "Reply ONLY with the detoxified sentence. Do not add any explanations or notes."
    )
    
    try:
        # Make the API call to Groq
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Toxic message: \"{comment}\""}
            ],
            model="llama3-8b-8192", 
            temperature=0.3, # Allow slight creativity for natural rephrasing
            max_tokens=50    # Keep the output concise
        )
        
        return chat_completion.choices[0].message.content.strip()
        
    except Exception as e:
        return "Error: Could not process request."

print("--- SMART LLM DETOXIFICATION RESULTS ---\n")

# Iterate through the test comments and generate context-aware detoxified versions
for comment in toxic_gaming_comments:
    print(f"Original Toxic : {comment}")
    
    # Generate the detoxified text using Llama-3
    result = smart_detoxify_with_llm(comment)
    
    print(f"Smart Detox    : {result}")
    print("-" * 60)

--- SMART LLM DETOXIFICATION RESULTS ---

Original Toxic : you are such a stupid noob, uninstall the game right now you absolute trash
Smart Detox    : Error: Could not process request.
------------------------------------------------------------
Original Toxic : stop feeding the enemy team you fucking idiot
Smart Detox    : Error: Could not process request.
------------------------------------------------------------
Original Toxic : my teammates are completely brain dead garbage
Smart Detox    : Error: Could not process request.
------------------------------------------------------------
Original Toxic : i swear to god i will kill you if you steal my loot again
Smart Detox    : Error: Could not process request.
------------------------------------------------------------
Original Toxic : shut up and play the game you pathetic loser
Smart Detox    : Error: Could not process request.
------------------------------------------------------------
